In [ ]:
import os
os.makedirs("Figs", exist_ok=True)  # figures are written to ./Figs

"""
Evaluate Kendall's tau vs. training fraction, reusing the ground-truth parsing and
build_per_protein of kendalltau_comparison_results_10seedens.ipynb. The prediction's own mask
already includes the tag/flag/4-sigma filters
(checked against tag_labels.csv: no leakage), so the manual BMRB 4-sigma / proline filtering
of the older version is dropped.

Data: Fig2/results/train_fraction_{method}/results/, method in
{panav, lacs, noref} (trainfrac10..90, plus trainfrac100 for PANAV; one seed / one replicate).
tau is a rank statistic and is insensitive to a constant referencing offset within a protein,
so all three methods are scored against the same GT (test_panav.json)
instead of a method-specific GT.
"""
import re
import json
import math
from collections import defaultdict
from pathlib import Path

import numpy as np
from scipy.stats import kendalltau


def parse_shift_list(lst, sequence, atom):
    vals, mask = [], []
    for i, x in enumerate(lst):
        residue = sequence[i]
        bad = False
        if x is None:
            bad = True
        elif isinstance(x, str):
            xs = x.strip().lower()
            if xs in {"none", "null", "nan", "inf", "+inf", "-inf"}:
                bad = True
            else:
                try:
                    x = float(x)
                except Exception:
                    bad = True
        if not bad and isinstance(x, (int, float)):
            if not math.isfinite(float(x)):
                bad = True
        if atom in ("N", "H") and residue == "P":
            bad = True
        if atom == "CB" and residue == "G":
            bad = True
        if bad:
            vals.append(0.0)
            mask.append(False)
        else:
            vals.append(float(x))
            mask.append(True)
    return vals, mask


def load_ground_truth(gt_path: Path, atom: str):
    """entry_ID (without the bmr prefix) -> (shift list, mask list)."""
    with open(gt_path) as f:
        rbmrb = json.load(f)
    out = {}
    for ref_id, shift_lst in rbmrb[atom].items():
        if all(x is None or (isinstance(x, str) and x.lower() in ("none", "null"))
               for x in shift_lst):
            continue
        sequence = rbmrb["sequence"][ref_id]
        shift, mask = parse_shift_list(shift_lst, sequence, atom=atom)
        entry = rbmrb["entry_ID"][ref_id].lower().removeprefix("bmr")
        out[entry] = (shift, mask)
    return out


def load_pred(path):
    """train_fraction prediction files are single-seed / single-replicate {"pred", "mask"} dicts
    (not the mean/std layout of the 10-seed ensemble); read them per entry directly."""
    with open(path) as f:
        d = json.load(f)
    return d["pred"], d["mask"]


def build_per_protein(pred_path, gt_path, atom):
    """entry -> {"pred": ..., "gt": ...}; intersects the prediction's own mask with the GT None/Pro-N-H mask only.
    tag/flag/4-sigma filters are already part of that mask, so they are not applied again here."""
    pred_dict, mask_dict = load_pred(pred_path)
    gt = load_ground_truth(Path(gt_path), atom=atom)

    per_protein = {}
    skipped = []
    for entry, (true_shift, true_mask) in gt.items():
        if entry not in pred_dict:
            skipped.append(entry)
            continue
        pred_shift = pred_dict[entry]
        pred_mask = mask_dict[entry]
        n = min(len(true_shift), len(true_mask), len(pred_shift), len(pred_mask))
        combined_mask = np.array(true_mask[:n], dtype=bool) & np.array(pred_mask[:n], dtype=bool)
        if not combined_mask.any():
            skipped.append(entry)
            continue
        per_protein[entry] = {
            "pred": np.asarray(pred_shift[:n], dtype=float)[combined_mask],
            "gt":   np.asarray(true_shift[:n], dtype=float)[combined_mask],
        }
    return per_protein, skipped


def taus_for_prediction(pred_path, gt_path, atom="N", min_valid_res=3):
    per_protein, skipped = build_per_protein(pred_path, gt_path, atom)
    taus = []
    for entry, d in per_protein.items():
        if len(d["gt"]) < min_valid_res:
            continue
        tau = kendalltau(d["pred"], d["gt"]).correlation
        if tau is not None and not np.isnan(tau):
            taus.append(tau)
    return taus, len(skipped)


RESULTS_DIR = Path("../../data/analysis/Fig2/results")
ATOM = "N"
MIN_VALID_RES = 3   # Kendall's tau is meaningless for very few residues; skip those proteins

METHODS = ["panav", "lacs", "noref", "noref_whole"]
GT_PATH = RESULTS_DIR / "test_panav.json"   # the PANAV test set is the GT for every method

# results_by_method[method][frac_pct] = (mean_tau, std_tau, n_entries)
results_by_method = {}

for method in METHODS:
    sample_dir = RESULTS_DIR / f"train_fraction_{method}" / "results"
    fname_re = re.compile(rf"predictions_N_{method}_.*_trainfrac(\d+)_.*\.json$")

    files_by_frac = defaultdict(list)
    for pth in sorted(sample_dir.glob(f"predictions_N_{method}_*_trainfrac*.json")):
        m = fname_re.match(pth.name)
        if not m:
            continue
        files_by_frac[int(m.group(1))].append(pth)

    fracs = sorted(files_by_frac)
    print(f"{method.upper()} training fraction (%): {fracs}")

    results = {}
    print(f"\n{'frac':>5s} {'mean_tau':>9s} {'std':>7s} {'n_ent':>6s} {'skip':>5s}")
    print("-" * 34)
    for frac_pct in fracs:
        all_taus, total_skipped = [], 0
        for pred_path in files_by_frac[frac_pct]:
            taus, skipped = taus_for_prediction(pred_path, GT_PATH, ATOM, MIN_VALID_RES)
            all_taus += taus
            total_skipped += skipped
        mean, std, n = np.mean(all_taus), np.std(all_taus), len(all_taus)
        results[frac_pct] = (mean, std, n)
        print(f"{frac_pct/100:5.1f} {mean:9.4f} {std:7.4f} {n:6d} {total_skipped:5d}")

    print("\n=== Marginal gain in mean tau vs previous fraction (saturation check) ===")
    print("     " + "".join(f"{f/100:>8.1f}" for f in fracs[1:]))
    row = f"{method.upper():7s}"
    for i in range(1, len(fracs)):
        d = results[fracs[i]][0] - results[fracs[i - 1]][0]
        row += f"{d:8.4f}"
    print(row)
    print()

    results_by_method[method] = results


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def set_style(base=12, family="Arial"):
    plt.rcParams.update({
        "font.family": family,
        "font.size": base,
        "axes.titlesize": base + 2,
        "axes.labelsize": base,
        "xtick.labelsize": base - 2,
        "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2,
        'svg.fonttype': 'none',
    })

set_style(base=19)

COLORS = {"panav": "#61AFEF", "lacs": "#E06C75", "noref": "#999999", "noref_whole": "#333333"}
LABELS = {"panav": "SLAE-CS-PANAV", "lacs": "SLAE-CS-LACS", "noref": "SLAE-CS-BASELINE", "noref_whole": "SLAE-CS-whole BMRB"}
YLIM = (0.65, 0.9)

fig, axes = plt.subplots(len(METHODS), 1, figsize=(8, 7), sharex=True)

for ax, method in zip(axes, METHODS):
    results = results_by_method[method]
    fracs_m = sorted(results)
    fracs = np.array(fracs_m) / 100.0
    means = np.array([results[f][0] for f in fracs_m])
    stds  = np.array([results[f][1] for f in fracs_m])
    color = COLORS[method]

    ax.plot(
        fracs, means, marker="o", markersize=5, lw=2.2, color=color,
        # label=f"{LABELS[method]}  (\u03c4@{fracs[-1]:.1f}={means[-1]:.4f})",
        label=f"{LABELS[method]}"
    )
    ax.fill_between(fracs, means - stds, means + stds, color=color, alpha=0.3, lw=0)

    ax.set_ylim(*YLIM)
    # if method == METHODS[1]:
    #     ax.set_ylabel("Mean Kendall's tau")
    # ax.legend(frameon=False, loc="lower right")
    # ax.grid(alpha=0.3, linestyle="--")

axes[-1].set_xlabel("Training fraction")
axes[-1].set_xlim(0.0, 1.05)

plt.tight_layout()
top = axes[0].get_position().y1
bottom = axes[-1].get_position().y0
p0 = axes[0].get_position()
axes[0].set_ylabel("Kendall's τ (mean ± std)")
axes[0].yaxis.set_label_coords(-0.07, ((top + bottom) / 2 - p0.y0) / p0.height)

handles = [ax.get_lines()[0] for ax in axes]
labels = [LABELS[m] for m in METHODS]
# fig.legend(
#     handles, labels,
#     loc="lower center", bbox_to_anchor=(0.5, 0.95),
#     ncol=2, frameon=False,
#     handlelength=1.5, columnspacing=1.0, handletextpad=0.4,
# )

plt.savefig("Figs/kendalltau_saturation_all_methods.png", dpi=300, bbox_inches="tight")
plt.savefig("Figs/kendalltau_saturation_all_methods.svg", bbox_inches="tight")
plt.show()

for method in METHODS:
    results = results_by_method[method]
    print(f"{LABELS[method]} tau: " + "  ".join(f"{f/100:.1f}={results[f][0]:.4f}" for f in sorted(results)))
